# Format NSCLC CosMx spatial data
Combine every slice's exprMat, per-slice metadata, and study-wide cell annotations into one sparse AnnData object.

In [1]:
from pathlib import Path
import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse

spatial_dir = Path('/work/magroup/skrieger/Syndri/data/datasets/nsclc_cosmx/raw/spatial')
study_metadata_file = spatial_dir / 'NSCLC_ST_cell_metadata.csv'
output_file = spatial_dir / 'st_concat.h5ad'
chunk_size = 5_000
slice_dirs = sorted(p for p in spatial_dir.iterdir() if p.is_dir() and (p / f'{p.name}-Flat_files_and_images').is_dir())
assert slice_dirs, f'No slice directories found in {spatial_dir}'
[p.name for p in slice_dirs]

['Lung12',
 'Lung13',
 'Lung5_Rep1',
 'Lung5_Rep2',
 'Lung5_Rep3',
 'Lung6',
 'Lung9_Rep1',
 'Lung9_Rep2']

In [2]:
# study_cell_ID has the form c_<slide>_<fov>_<cell>; its final token is
# the cell_ID used in each exprMat and per-slice metadata table.
study_metadata = pd.read_csv(study_metadata_file, low_memory=False).rename(columns={'cell_ID': 'study_cell_ID', 'tissue': 'study_tissue'})
study_metadata['cell_ID'] = pd.to_numeric(study_metadata['study_cell_ID'].str.rsplit('_', n=1).str[-1], errors='raise').astype(np.int64)
study_metadata['fov'] = pd.to_numeric(study_metadata['fov'], errors='raise').astype(np.int64)
study_keys = ['Run_Tissue_name', 'fov', 'cell_ID']
assert not study_metadata.duplicated(study_keys).any(), f'Duplicate keys in {study_metadata_file}'
study_metadata.groupby('Run_Tissue_name', observed=True).size()

Run_Tissue_name
Lung12         71304
Lung13         81236
Lung5_Rep1     98002
Lung5_Rep2    105800
Lung5_Rep3     97809
Lung6          89975
Lung9_Rep1     87606
Lung9_Rep2    139504
dtype: int64

In [3]:
def load_cosmx_slice(slice_dir, study_metadata, chunk_size=5_000):
    slice_name = slice_dir.name
    flat_dir = slice_dir / f'{slice_name}-Flat_files_and_images'
    expr_file = flat_dir / f'{slice_name}_exprMat_file.csv'
    metadata_file = flat_dir / f'{slice_name}_metadata_file.csv'
    if not expr_file.is_file() or not metadata_file.is_file():
        raise FileNotFoundError(f'Missing exprMat or metadata for {slice_name}')

    matrices, obs_chunks, gene_names = [], [], None
    for chunk in pd.read_csv(expr_file, chunksize=chunk_size):
        chunk['fov'] = pd.to_numeric(chunk['fov'], errors='raise').astype(np.int64)
        chunk['cell_ID'] = pd.to_numeric(chunk['cell_ID'], errors='raise').astype(np.int64)
        current_genes = chunk.columns.drop(['fov', 'cell_ID']).tolist()
        if gene_names is None:
            gene_names = current_genes
        elif current_genes != gene_names:
            raise ValueError(f'Gene columns changed within {expr_file}')
        obs_chunks.append(chunk[['fov', 'cell_ID']].copy())
        values = chunk[gene_names].to_numpy(dtype=np.int32, copy=False)
        if (values < 0).any():
            raise ValueError(f'Negative counts found in {expr_file}')
        matrices.append(sparse.csr_matrix(values))

    X = sparse.vstack(matrices, format='csr')
    obs = pd.concat(obs_chunks, ignore_index=True)
    keys = ['fov', 'cell_ID']
    if obs.duplicated(keys).any():
        raise ValueError(f'Duplicate expression keys in {expr_file}')

    slice_metadata = pd.read_csv(metadata_file, low_memory=False)
    for key in keys:
        slice_metadata[key] = pd.to_numeric(slice_metadata[key], errors='raise').astype(np.int64)
    if slice_metadata.duplicated(keys).any():
        raise ValueError(f'Duplicate metadata keys in {metadata_file}')
    slice_metadata['_has_slice_metadata'] = True

    # Left joins retain every exprMat row. Overlapping fields from the large
    # study table receive a _study suffix; per-slice fields keep their names.
    obs['_expr_order'] = np.arange(len(obs))
    obs = obs.merge(slice_metadata, on=keys, how='left', validate='one_to_one', sort=False)
    slice_study = study_metadata.loc[study_metadata['Run_Tissue_name'].eq(slice_name)]
    obs = obs.merge(slice_study, on=keys, how='left', validate='one_to_one', sort=False, suffixes=('', '_study'))
    obs = obs.sort_values('_expr_order').drop(columns='_expr_order').reset_index(drop=True)
    obs.insert(0, 'tissue', slice_name)
    obs.index = pd.Index(slice_name + ':' + obs['fov'].astype(str) + ':' + obs['cell_ID'].astype(str), name='cell_key')
    assert obs.index.is_unique and len(obs) == X.shape[0]
    obs['_has_slice_metadata'] = obs['_has_slice_metadata'].eq(True)
    for column in obs.select_dtypes(include='object'):
        obs[column] = obs[column].fillna('').astype(str)

    adata = ad.AnnData(X=X, obs=obs, var=pd.DataFrame(index=pd.Index(gene_names, name='gene')))
    adata.uns['source_exprMat_file'] = str(expr_file)
    adata.uns['source_slice_metadata_file'] = str(metadata_file)
    adata.uns['source_study_metadata_file'] = str(study_metadata_file)
    return adata

In [4]:
slice_adatas = []
for slice_dir in slice_dirs:
    print(f'Loading {slice_dir.name} ...', flush=True)
    slice_adata = load_cosmx_slice(slice_dir, study_metadata, chunk_size)
    print(f"  {slice_adata.n_obs:,} cells x {slice_adata.n_vars:,} genes; {slice_adata.obs['cell_type'].ne('').sum():,} study annotations")
    slice_adatas.append(slice_adata)

# Check the ordered panel explicitly before using AnnData's supported
# inner join, so intersection cannot silently hide a panel mismatch.
expected_var_names = slice_adatas[0].var_names
assert all(a.var_names.equals(expected_var_names) for a in slice_adatas), 'Ordered gene panels differ between slices'
st_data = ad.concat(slice_adatas, axis=0, join='inner', merge='same', uns_merge='unique', index_unique=None)
assert st_data.var_names.equals(expected_var_names)
assert st_data.obs_names.is_unique
st_data.uns['source_study_metadata_file'] = str(study_metadata_file)
st_data.uns['source_exprMat_files'] = [a.uns['source_exprMat_file'] for a in slice_adatas]
st_data.uns['source_slice_metadata_files'] = [a.uns['source_slice_metadata_file'] for a in slice_adatas]
st_data

Loading Lung12 ...
  74,025 cells x 980 genes; 71,304 study annotations
Loading Lung13 ...
  82,863 cells x 980 genes; 81,236 study annotations
Loading Lung5_Rep1 ...
  100,322 cells x 980 genes; 98,002 study annotations
Loading Lung5_Rep2 ...
  106,689 cells x 980 genes; 105,800 study annotations
Loading Lung5_Rep3 ...
  100,294 cells x 980 genes; 97,809 study annotations
Loading Lung6 ...
  93,825 cells x 980 genes; 89,975 study annotations
Loading Lung9_Rep1 ...
  91,992 cells x 980 genes; 87,606 study annotations
Loading Lung9_Rep2 ...
  150,549 cells x 980 genes; 139,504 study annotations


AnnData object with n_obs × n_vars = 800559 × 980
    obs: 'tissue', 'fov', 'cell_ID', 'Area', 'AspectRatio', 'CenterX_local_px', 'CenterY_local_px', 'CenterX_global_px', 'CenterY_global_px', 'Width', 'Height', 'Mean.MembraneStain', 'Max.MembraneStain', 'Mean.PanCK', 'Max.PanCK', 'Mean.CD45', 'Max.CD45', 'Mean.CD3', 'Max.CD3', 'Mean.DAPI', 'Max.DAPI', '_has_slice_metadata', 'sdimx', 'sdimy', 'study_cell_ID', 'Area_study', 'AspectRatio_study', 'Width_study', 'Height_study', 'Mean.CD298', 'Max.CD298', 'Mean.G', 'Max.G', 'Mean.Y', 'Max.Y', 'Mean.R', 'Max.R', 'Mean.DAPI_study', 'Max.DAPI_study', 'dualfiles', 'Slide_name', 'study_tissue', 'Run_name', 'ISH.concentration', 'Dash', 'slide_ID_numeric', 'Run_Tissue_name', 'Panel', 'Diversity', 'totalcounts', 'log10totalcounts', 'background', 'remove_flagged_cells', 'patient', 'cell_type', 'niche', 'prop_tumor_in_100_neighbors'
    uns: 'source_study_metadata_file', 'source_exprMat_files', 'source_slice_metadata_files'

In [5]:
summary = (st_data.obs.assign(
    has_slice_metadata=st_data.obs['_has_slice_metadata'],
    has_study_annotation=st_data.obs['cell_type'].ne(''),
).groupby('tissue', observed=True).agg(
    cells=('cell_ID', 'size'),
    cells_with_slice_metadata=('has_slice_metadata', 'sum'),
    cells_with_study_annotation=('has_study_annotation', 'sum'),
))
display(summary)
assert st_data.X.shape == (len(st_data.obs), len(st_data.var))
assert np.isfinite(st_data.X.data).all() and (st_data.X.data >= 0).all()
st_data.write_h5ad(output_file, compression='gzip')
print(f'Wrote {st_data.n_obs:,} cells x {st_data.n_vars:,} genes to {output_file}')

,cells,cells_with_slice_metadata,cells_with_study_annotation
tissue,,,
Lung12,74025,73997,71304
Lung13,82863,82843,81236
Lung5_Rep1,100322,100292,98002
Lung5_Rep2,106689,106660,105800
Lung5_Rep3,100294,100264,97809
Lung6,93825,93795,89975
Lung9_Rep1,91992,91972,87606
Lung9_Rep2,150549,150504,139504


Wrote 800,559 cells x 980 genes to /work/magroup/skrieger/Syndri/data/datasets/nsclc_cosmx/raw/spatial/st_concat.h5ad
